# CPE15 - Week 2: NumPy Arrays and Vectorized Computation

**Programming for Data Science | Professional Elective 1 | AY 2026-2027**

| Syllabus element | Alignment |
|---|---|
| Course outcome | CO2 |
| Course objective | COBJ1 |
| Assessment connection | Quiz 2 and NumPy activity |
| Sustainable Development Goals | 4, 9 |

This notebook is designed for explanation, live coding, guided practice, and independent follow-through. Run it from top to bottom in a fresh kernel so that every result can be reproduced.


## Lecture map

1. Why NumPy arrays?
2. Creating arrays and inspecting their structure
3. Indexing, slicing, and views
4. Boolean selection
5. Reshaping and axis meaning
6. Broadcasting and vectorized computation
7. Case study: power and quality calculations

    **Live-teaching rhythm:** define the idea → predict the result → run a focused example → inspect the saved output → explain the evidence → complete the practice task.

## Learning outcomes

    By the end of the session, students should be able to:

- create NumPy arrays with appropriate shapes and data types;
- retrieve and update values through indexing, slicing, and Boolean masks;
- reshape arrays without losing track of observation and feature meaning;
- apply broadcasting and vectorized operations to engineering measurements;
- verify array results using shape, dtype, and range checks.

    ## Lecture route

1. Why arrays differ from Python lists
2. Creation, properties, indexing, and slicing
3. Boolean selection and reshaping
4. Broadcasting and vectorized computation
5. A multi-sensor calibration activity


In [ ]:
from pathlib import Path
import platform
import random
import sys

random.seed(15)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.system()}")
print(f"Artifacts folder: {ARTIFACTS.resolve()}")


> **Reproducibility habit:** a notebook is not finished merely because it ran once. It should run in order from a restarted kernel, use explicit inputs, avoid hidden state, and explain the meaning of its outputs.


## 1. Why NumPy arrays?

A NumPy array stores elements in a regular, typed structure. This enables concise operations over entire vectors and matrices. A Python list is flexible and may contain mixed objects; an array is deliberately more regular. That regularity supports numerical speed, predictable shape behavior, and a large scientific-computing ecosystem.

Vectorization means describing the operation on a whole array instead of writing a Python loop for every element.


In [ ]:
import numpy as np

python_values = [1, 2, 3]
numpy_values = np.array([1, 2, 3])

print("List * 2:", python_values * 2)
print("Array * 2:", numpy_values * 2)
print("NumPy version:", np.__version__)


### Additional worked case — same symbols, different data models

A Python list treats `* 2` as sequence repetition, so `[1, 2, 3] * 2` contains six items. A NumPy array treats the same symbol as elementwise numerical multiplication. This contrast matters because code can run successfully while expressing the wrong model. Also compare adding two lists, which concatenates them, with adding two arrays, which adds aligned elements. Array arithmetic requires compatible shapes and usually a common numerical dtype.

In [ ]:
clinic_list_a = [1, 2, 3]
clinic_list_b = [10, 20, 30]
clinic_array_a = np.array(clinic_list_a)
clinic_array_b = np.array(clinic_list_b)

print("List addition:", clinic_list_a + clinic_list_b)
print("Array addition:", clinic_array_a + clinic_array_b)
print("Array square:", clinic_array_a ** 2)

**Result and interpretation.** List addition produces six positions because it concatenates sequences. Array addition produces `[11, 22, 33]`, and exponentiation produces `[1, 4, 9]`. Before using an operator, state whether the intended operation is structural or numerical.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** List addition produces six positions because it concatenates sequences.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Multiply the Python list `[1, 2, 3]` and the NumPy array made from it by 2; compare the two results and identify which behavior is numerical vectorization.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Why NumPy arrays?
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 2. Creating arrays and inspecting their structure

Always inspect `shape`, `ndim`, `size`, and `dtype` before serious computation. Shape expresses the organization of axes; dtype expresses how elements are represented. A two-dimensional array with shape `(4, 3)` might mean four observations and three sensor channels, but NumPy does not know that meaning unless you document it.


In [ ]:
readings = np.array(
    [
        [3.28, 0.41, 28.5],
        [3.31, 0.39, 28.7],
        [3.26, 0.44, 29.0],
        [3.30, 0.40, 28.8],
    ],
    dtype=float,
)

{
    "shape": readings.shape,
    "dimensions": readings.ndim,
    "elements": readings.size,
    "dtype": str(readings.dtype),
}


### Array constructors for different starting conditions

The previous cell built an array from observed values. This continuation compares constructors used when values follow a pattern: `zeros` and `ones` initialize placeholders, `arange` uses a start-stop-step rule, `linspace` requests an exact number of evenly spaced values, and `eye` creates an identity matrix. Their shapes and dtypes should be inspected before later assignment.

In [ ]:
zeros = np.zeros((2, 3))
ones = np.ones((2, 3))
sequence = np.arange(0, 12, 2)
grid = np.linspace(0, 1, 5)
identity = np.eye(3)

print("arange:", sequence)
print("linspace:", grid)
print("identity:", identity, sep=chr(10))


Use `arange` when the step is central and `linspace` when the required number of evenly spaced samples is central. With floating-point steps, `linspace` is often easier to reason about at the endpoint.


### Discussion clinic — shape, dimension, size, and dtype answer different questions

For a matrix-shaped array, `shape` reports the length of each axis, `ndim` reports the number of axes, and `size` reports the total number of elements. A `(4, 3)` array therefore has two dimensions and twelve elements. `dtype` describes how every element is stored. Mixed integers and decimals normally promote to a floating dtype; mixing numbers with text can promote the entire array to strings, which blocks ordinary numerical summaries.

**Interpretation standard.** When describing an array, say more than 'it is a matrix.' State what one row represents, what each column represents, its shape, and its dtype. Those facts determine whether later slicing, broadcasting, and aggregation are meaningful.

### 2.1 Individual topic — `np.array` and `dtype`

**What it is and how it works.** `np.array` converts a compatible Python sequence into a homogeneous n-dimensional array. NumPy selects one storage dtype for all elements; `dtype=` requests a specific representation.

**Core syntax**

```python
`np.array(values, dtype=float)`
```

**When to use it.** Use it when measurements need numerical operations, shape-aware indexing, or vectorization.

**When to use another approach.** Do not force incompatible labels and measurements into one array; a DataFrame or structured record is clearer.

In [ ]:
# Demonstration — `np.array` and `dtype`
topic_integer_array = np.array([1, 2, 3])
topic_float_array = np.array([1, 2, 3], dtype=float)
print(topic_integer_array, topic_integer_array.dtype)
print(topic_float_array, topic_float_array.dtype)

**Expected output pattern and interpretation.** Both arrays contain the same visible magnitudes, but one stores integers and the other floating-point values. Dtype affects memory, precision, and valid operations.

**Science-communication statement.** Say which values were converted, the resulting dtype, and why that dtype is appropriate; do not describe dtype conversion as improved measurement accuracy.

### 2.2 Individual topic — `shape`, `ndim`, and `size`

**What it is and how it works.** These attributes answer different structural questions: `shape` gives each axis length, `ndim` counts axes, and `size` counts all elements.

**Core syntax**

```python
`array.shape`, `array.ndim`, `array.size`
```

**When to use it.** Use them before slicing, reshaping, broadcasting, aggregation, or reporting a table-like array.

**When to use another approach.** Do not say only that an array is 'two-dimensional'; define what its rows and columns represent.

In [ ]:
# Demonstration — `shape`, `ndim`, and `size`
topic_structure = np.arange(12).reshape(4, 3)
print("shape:", topic_structure.shape)
print("ndim:", topic_structure.ndim)
print("size:", topic_structure.size)

**Expected output pattern and interpretation.** The array has four rows, three columns, two axes, and twelve total elements. The product of the shape dimensions equals size.

**Science-communication statement.** Report `(4 observations, 3 variables)` rather than only `(4, 3)` when the real-world meanings are known.

### 2.3 Individual topic — `np.zeros` and `np.ones`

**What it is and how it works.** These constructors create initialized arrays of a requested shape. The shape argument is a tuple; dtype defaults to floating point unless specified.

**Core syntax**

```python
`np.zeros((rows, columns))`; `np.ones(shape, dtype=int)`
```

**When to use it.** Use them for clearly defined initialization, masks, counters, or arrays that will be filled later.

**When to use another approach.** Do not use zeros as substitutes for missing observations unless zero is a valid measured value.

In [ ]:
# Demonstration — `np.zeros` and `np.ones`
topic_zero_grid = np.zeros((2, 3))
topic_one_flags = np.ones(4, dtype=int)
print(topic_zero_grid, sep=chr(10))
print(topic_one_flags)

**Expected output pattern and interpretation.** The first result has two rows and three floating zeros; the second has four integer ones. These are initialized values, not collected data.

**Science-communication statement.** Call them placeholders or initial values, and distinguish them explicitly from observed zeros or measured ones.

### 2.4 Individual topic — `np.arange` and `np.linspace`

**What it is and how it works.** `arange` follows a start-stop-step rule and excludes the stop. `linspace` requests an exact number of evenly spaced values and normally includes both endpoints.

**Core syntax**

```python
`np.arange(start, stop, step)`; `np.linspace(start, stop, num)`
```

**When to use it.** Use `arange` for integer-like steps and `linspace` for a specified sample count over a continuous interval.

**When to use another approach.** Avoid relying on exact endpoint behavior from floating-point `arange`; `linspace` is safer for fixed endpoints.

In [ ]:
# Demonstration — `np.arange` and `np.linspace`
topic_steps = np.arange(0, 10, 2)
topic_samples = np.linspace(0, 10, 6)
print("arange:", topic_steps)
print("linspace:", topic_samples)

**Expected output pattern and interpretation.** Both produce `[0, 2, 4, 6, 8]`-like spacing, but `linspace` includes 10 because six samples including both endpoints were requested.

**Science-communication statement.** State whether the design fixed the step size or the number of samples, because they imply different experimental choices.

### 2.5 Individual topic — `np.eye` identity matrix

**What it is and how it works.** `np.eye(n)` creates a square matrix with ones on the main diagonal and zeros elsewhere. Multiplying by it preserves a compatible vector or matrix.

**Core syntax**

```python
`np.eye(n, dtype=float)`
```

**When to use it.** Use it for identity checks, transformations, initialization, and demonstrations of matrix properties.

**When to use another approach.** Do not confuse an identity matrix with an all-ones matrix or use it as evidence that an unrelated model is correct.

In [ ]:
# Demonstration — `np.eye` identity matrix
topic_identity = np.eye(3)
topic_vector = np.array([4.0, 5.0, 6.0])
print(topic_identity, sep=chr(10))
print("I @ vector:", topic_identity @ topic_vector)

**Expected output pattern and interpretation.** The matrix product returns the original vector, demonstrating the identity property in three dimensions.

**Science-communication statement.** Describe this as a numerical identity check; it verifies the operation, not the scientific meaning of the vector.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** When describing an array, say more than 'it is a matrix.' State what one row represents, what each column represents, its shape, and its dtype.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Create a 2 × 3 floating-point array of voltage readings and report its `shape`, `ndim`, `size`, and `dtype`, including what each axis represents.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Creating arrays and inspecting their structure
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 3. Indexing, slicing, and views

Indexing retrieves individual elements or lower-dimensional selections. Slicing uses `start:stop:step`, with the stop excluded. NumPy slices are frequently **views** into the original data, so changing a slice may change the source array. Use `.copy()` when independent data are required.


In [ ]:
voltage = readings[:, 0]
current = readings[:, 1]
first_two_rows = readings[:2, :]
temperature_copy = readings[:, 2].copy()

print("Voltage:", voltage)
print("First two rows:", first_two_rows, sep=chr(10))
print("Copied temperature:", temperature_copy)


### View mutation demonstration

The earlier slice selected columns from the measurement array and deliberately copied temperature. This smaller experiment shows why `.copy()` matters: `view = demo[1:4]` shares memory with `demo`. Assigning `view[0] = 99` changes the element at index 1 of the original array. The displayed source mutation is the lesson, not an accidental side effect.

In [ ]:
demo = np.arange(6)
view = demo[1:4]
view[0] = 99
print("View:", view)
print("Original changed:", demo)


### Discussion clinic — a slice may share memory while a copy does not

Basic NumPy slices are usually **views** into the original array. Changing a view can therefore change the source. Calling `.copy()` creates independent storage. This is different from merely assigning another name, which never copies the data. Use `np.shares_memory(a, b)` when memory sharing matters, and make a deliberate copy before destructive cleaning if the original measurement array must remain available for audit.

**Interpretation standard.** Indexing answers 'which element?', slicing answers 'which rectangular region?', and copying answers 'should later mutation remain isolated?' Treat those as three separate design decisions.

### 3.1 Individual topic — Integer indexing

**What it is and how it works.** Integer indexing selects one position along each specified axis. Indices start at zero, and negative indices count backward from the end.

**Core syntax**

```python
`array[row, column]`; `array[-1]`
```

**When to use it.** Use it when the exact position is meaningful and known.

**When to use another approach.** Avoid positional selection when the requirement is label-based or when row order can change.

In [ ]:
# Demonstration — Integer indexing
topic_index_grid = np.arange(12).reshape(3, 4)
print("row 1, column 2:", topic_index_grid[1, 2])
print("last row:", topic_index_grid[-1])

**Expected output pattern and interpretation.** Index `(1, 2)` returns 6 because it refers to the second row and third column. `-1` returns the final row.

**Science-communication statement.** Translate the position into domain language, such as 'second observation, third channel,' rather than reporting only an index.

### 3.2 Individual topic — Slicing

**What it is and how it works.** A slice uses `start:stop:step`; the stop position is excluded. In multiple dimensions, one slice can be supplied per axis.

**Core syntax**

```python
`array[row_start:row_stop, column_start:column_stop]`
```

**When to use it.** Use it for contiguous ranges or regularly stepped subsets.

**When to use another approach.** Do not assume the stop value is included, and do not omit the axis meaning from the explanation.

In [ ]:
# Demonstration — Slicing
topic_slice_grid = np.arange(20).reshape(4, 5)
topic_block = topic_slice_grid[1:3, 2:5]
print(topic_slice_grid, sep=chr(10))
print("selected block:", topic_block, sep=chr(10))

**Expected output pattern and interpretation.** Rows 1 and 2 and columns 2 through 4 are retained, so the block shape is `(2, 3)`.

**Science-communication statement.** Report both the selection rule and resulting shape so readers can verify what portion of the data remains.

### 3.3 Individual topic — View

**What it is and how it works.** A basic NumPy slice usually returns a view that shares the original memory. Mutating the view can therefore mutate the source array.

**Core syntax**

```python
`view = array[start:stop]`; `np.shares_memory(array, view)`
```

**When to use it.** Use views for memory-efficient read-only work or deliberate in-place updates.

**When to use another approach.** Avoid unnoticed view mutation when the original evidence must remain unchanged.

In [ ]:
# Demonstration — View
topic_source = np.array([10, 20, 30, 40])
topic_view = topic_source[1:3]
print("shares memory:", np.shares_memory(topic_source, topic_view))
topic_view[0] = 999
print("source after view mutation:", topic_source)

**Expected output pattern and interpretation.** The source changes from 20 to 999 because the view shares storage with positions 1 and 2.

**Science-communication statement.** Disclose that the operation modified the original array; mutation history matters for reproducibility.

### 3.4 Individual topic — Copy

**What it is and how it works.** `.copy()` allocates independent storage containing the same current values. Later mutation of the copy does not change the source.

**Core syntax**

```python
`independent = array_slice.copy()`
```

**When to use it.** Use it before destructive cleaning, experimentation, or transformations that must preserve raw evidence.

**When to use another approach.** Avoid unnecessary large copies when a documented read-only view is sufficient.

In [ ]:
# Demonstration — Copy
topic_source_copy = np.array([10, 20, 30, 40])
topic_independent = topic_source_copy[1:3].copy()
topic_independent[0] = 999
print("copy:", topic_independent)
print("source preserved:", topic_source_copy)

**Expected output pattern and interpretation.** Only the copied array changes; the original retains 20 at index 1. `np.shares_memory` would be false.

**Science-communication statement.** Say that an independent working copy was created to preserve the original values; do not imply copying validates them.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** Indexing answers 'which element?', slicing answers 'which rectangular region?', and copying answers 'should later mutation remain isolated?' Treat those as three separate design decisions.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Take a slice from an array, mutate the slice, and observe the source; repeat with `.copy()` and explain the difference between a view and an independent array.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Indexing, slicing, and views
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 4. Boolean selection

A Boolean mask has the same relevant shape as the data being filtered. Combine elementwise conditions with `&`, `|`, and `~`, and parenthesize each comparison. Python's scalar `and` and `or` do not perform elementwise array logic.


In [ ]:
valid_voltage_mask = (voltage >= 3.25) & (voltage <= 3.35)
high_current_mask = current > 0.42

print("Valid voltage rows:", readings[valid_voltage_mask], sep=chr(10))
print("High-current row indices:", np.where(high_current_mask)[0])


### Additional worked case — compose masks and reconcile retained rows

A Boolean mask contains one `True` or `False` per candidate row. Parentheses are required around comparisons combined with `&` or `|` because these are elementwise operators. The complement `~mask` selects rejected items. Counting `mask.sum()` and `(~mask).sum()` provides a simple reconciliation check: retained plus rejected must equal the source row count.

In [ ]:
clinic_temperature = readings[:, 2]
clinic_mask = valid_voltage_mask & (clinic_temperature < 28.9)

print("Combined mask:", clinic_mask)
print("Retained rows:", readings[clinic_mask], sep=chr(10))
print("Retained / rejected:", int(clinic_mask.sum()), int((~clinic_mask).sum()))
assert int(clinic_mask.sum() + (~clinic_mask).sum()) == len(readings)

**Result and interpretation.** The combined mask keeps only rows satisfying both the voltage interval and the temperature limit. The assertion proves that every row was classified exactly once; it does not prove that the chosen limits are scientifically justified.

### 4.1 Individual topic — Boolean mask

**What it is and how it works.** A comparison against an array produces a Boolean array with one truth value per candidate element. Using that mask as an index retains values where the mask is true.

**Core syntax**

```python
`mask = values >= limit`; `selected = values[mask]`
```

**When to use it.** Use it for transparent value-based filtering.

**When to use another approach.** Do not report only selected values; preserve the rule and rejected count.

In [ ]:
# Demonstration — Boolean mask
topic_voltage = np.array([3.10, 3.25, 3.30, 3.60])
topic_valid_mask = topic_voltage.between(3.2, 3.4) if hasattr(topic_voltage, "between") else ((topic_voltage >= 3.2) & (topic_voltage <= 3.4))
print("mask:", topic_valid_mask)
print("selected:", topic_voltage[topic_valid_mask])

**Expected output pattern and interpretation.** Only 3.25 and 3.30 satisfy the inclusive interval, so two of four values are retained.

**Science-communication statement.** State the interval, inclusivity, and retained fraction; avoid calling excluded values faulty without a justified standard.

### 4.2 Individual topic — Combining masks

**What it is and how it works.** Elementwise `&`, `|`, and `~` combine or invert Boolean arrays. Each comparison must be parenthesized because operator precedence differs from plain-language reading.

**Core syntax**

```python
`(a >= low) & (a <= high)`; `mask_a | mask_b`; `~mask`
```

**When to use it.** Use combined masks when every selection condition should remain explicit and inspectable.

**When to use another approach.** Do not use scalar `and` or `or` with arrays; their single-truth-value expectation is ambiguous.

In [ ]:
# Demonstration — Combining masks
topic_temp = np.array([25.0, 31.0, 35.0, 40.0])
topic_humidity = np.array([60, 82, 70, 90])
topic_alert = (topic_temp >= 35) & (topic_humidity >= 75)
print("alert mask:", topic_alert)
print("alert rows:", np.column_stack([topic_temp, topic_humidity])[topic_alert])

**Expected output pattern and interpretation.** Only the 40 °C and 90% row satisfies both conditions. The 35 °C row fails because humidity is below 75%.

**Science-communication statement.** Explain each condition and the logical connector; readers should be able to reproduce why each row passed or failed.

### 4.3 Individual topic — `np.where`

**What it is and how it works.** With one argument, `np.where(condition)` returns matching indices. With three arguments, it selects one value when true and another when false.

**Core syntax**

```python
`np.where(mask)[0]`; `np.where(mask, value_if_true, value_if_false)`
```

**When to use it.** Use it for locating matches or vectorized two-way labeling.

**When to use another approach.** Avoid deeply nested `where` expressions for multi-class policy logic; explicit functions may communicate better.

In [ ]:
# Demonstration — `np.where`
topic_current = np.array([0.20, 0.55, 0.90])
topic_high = topic_current >= 0.50
print("matching indices:", np.where(topic_high)[0])
print("labels:", np.where(topic_high, "review", "normal"))

**Expected output pattern and interpretation.** Indices 1 and 2 match, and the corresponding labels are `review`; index 0 is `normal`.

**Science-communication statement.** Call these labels results of an illustrative threshold rule, not measured states or expert diagnoses.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** The combined mask keeps only rows satisfying both the voltage interval and the temperature limit.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Create separate valid-voltage and nonmissing-current masks, combine them with `&`, and display received, retained, and rejected row counts.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Boolean selection
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 5. Reshaping and axis meaning

Reshaping changes how elements are organized, not their order or total count. The product of the dimensions must remain equal to `size`. Use `-1` to let NumPy infer one dimension, but still verify the result.


In [ ]:
one_day = np.arange(24)
six_blocks = one_day.reshape(6, 4)
restored = six_blocks.reshape(-1)

print("Reshaped to six 4-hour blocks:", six_blocks, sep=chr(10))
assert np.array_equal(restored, one_day)


### Additional worked case — reshape changes organization, whereas an axis reduction changes granularity

`reshape` changes how the same elements are arranged and must preserve total size. An axis operation such as `mean(axis=0)` reduces rows and returns one result per column; `mean(axis=1)` reduces columns and returns one result per row. Always name the real-world meaning of each axis before selecting it. The shortcut `-1` asks NumPy to infer one dimension from the remaining dimensions and total element count.

In [ ]:
clinic_grid = np.arange(12).reshape(3, 4)
print("Grid:", clinic_grid, sep=chr(10))
print("One mean per column (axis=0):", clinic_grid.mean(axis=0))
print("One mean per row (axis=1):", clinic_grid.mean(axis=1))
print("Flattened again:", clinic_grid.reshape(-1))

**Result and interpretation.** The column reduction returns four means, the row reduction returns three, and flattening restores twelve positions. The result length is a useful clue for checking whether the intended axis was selected.

### 5.1 Individual topic — `reshape`

**What it is and how it works.** `reshape` changes axis lengths without changing element count or element order under the chosen memory order.

**Core syntax**

```python
`array.reshape(new_rows, new_columns)`
```

**When to use it.** Use it when the same sequence must be interpreted with a new documented dimensional structure.

**When to use another approach.** Do not reshape merely until code runs; the new axes must have a real meaning.

In [ ]:
# Demonstration — `reshape`
topic_hours = np.arange(12)
topic_blocks = topic_hours.reshape(3, 4)
print(topic_blocks, sep=chr(10))
print("sizes equal:", topic_hours.size == topic_blocks.size)

**Expected output pattern and interpretation.** Twelve values become three rows of four values, while total size remains twelve.

**Science-communication statement.** Name the new axes, such as 'three periods by four hourly samples,' and state that values were reorganized rather than newly observed.

### 5.2 Individual topic — Inferred dimension with `-1`

**What it is and how it works.** Exactly one reshape dimension may be `-1`; NumPy infers its length from total element count and the other dimensions.

**Core syntax**

```python
`array.reshape(-1, known_width)`
```

**When to use it.** Use it when one dimension is fixed and the other should be calculated safely.

**When to use another approach.** Do not use more than one `-1`, and still verify the inferred shape matches the intended grouping.

In [ ]:
# Demonstration — Inferred dimension with `-1`
topic_stream = np.arange(24)
topic_six_columns = topic_stream.reshape(-1, 6)
print("inferred shape:", topic_six_columns.shape)
print(topic_six_columns, sep=chr(10))

**Expected output pattern and interpretation.** NumPy infers four rows because 24 elements divided into columns of six yields four complete groups.

**Science-communication statement.** Report the inferred dimension and the grouping assumption; automatic inference is arithmetic, not domain validation.

### 5.3 Individual topic — Axis reduction

**What it is and how it works.** A reduction collapses the specified axis. For a `(rows, columns)` table, `axis=0` returns one result per column and `axis=1` one per row.

**Core syntax**

```python
`array.mean(axis=0)`; `array.sum(axis=1)`
```

**When to use it.** Use it for summaries whose denominator and retained dimension are defined.

**When to use another approach.** Avoid memorizing axis numbers without naming the row and column meanings.

In [ ]:
# Demonstration — Axis reduction
topic_measurements = np.array([[1, 2, 3], [4, 5, 6]])
print("column means:", topic_measurements.mean(axis=0))
print("row sums:", topic_measurements.sum(axis=1))

**Expected output pattern and interpretation.** Three column means are returned after rows are reduced; two row sums are returned after columns are reduced.

**Science-communication statement.** State what was averaged or summed, the denominator for each result, and the units after the operation.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** The column reduction returns four means, the row reduction returns three, and flattening restores twelve positions.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Reshape twelve hourly readings into a 3 × 4 array, calculate means along both axes, and label what one value from each reduction summarizes.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Reshaping and axis meaning
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 6. Broadcasting and vectorized computation

Broadcasting allows arrays with compatible shapes to interact without explicitly copying repeated values. Compare shapes from the trailing dimensions: dimensions are compatible when they are equal or one of them is 1.

The next dataset has rows as observations and columns as sensor channels. A length-three calibration vector therefore aligns with the columns.


In [ ]:
raw = np.array(
    [
        [510, 205, 720],
        [515, 198, 730],
        [505, 210, 710],
        [520, 202, 725],
    ],
    dtype=float,
)
scales = np.array([0.01, 0.05, 0.10])
offsets = np.array([-0.20, 1.00, -40.00])

calibrated = raw * scales + offsets
print("Raw shape:", raw.shape)
print("Scale shape:", scales.shape)
print("Calibrated:", np.round(calibrated, 2), sep=chr(10))


### Reducing the calibrated table along each axis

Calibration preserves the `(observations, channels)` shape. The next cell contrasts `axis=0`, which collapses observations and returns one mean per channel, with `axis=1`, which collapses channels and returns one mean per observation. The two result lengths should match the untouched dimension in each case.

In [ ]:
channel_means = calibrated.mean(axis=0)
observation_means = calibrated.mean(axis=1)

print("Mean per channel:", np.round(channel_means, 2))
print("Mean per observation:", np.round(observation_means, 2))


`axis=0` collapses rows and returns one result per column. `axis=1` collapses columns and returns one result per row. Translate axis operations into words before coding: "mean across observations for each channel" is clearer than memorizing a number.


### Discussion clinic — broadcasting aligns dimensions from the right

Broadcasting avoids manually repeating calibration factors. NumPy compares shapes from the trailing dimension: dimensions are compatible when they are equal or one of them is 1. Thus a `(4, 3)` measurement table can combine with a `(3,)` vector because the vector supplies one factor for each column. A `(4,)` vector would not mean one factor per row without reshaping it to `(4, 1)`.

**Interpretation standard.** Broadcasting is not magic duplication; it is a documented alignment rule. Write down both operand shapes and label what each dimension means before relying on an implicit expansion.

### 6.1 Individual topic — Broadcasting

**What it is and how it works.** Broadcasting aligns shapes from the right and virtually expands dimensions that are equal or length one. It avoids manually repeating compatible factors.

**Core syntax**

```python
`matrix * column_factors`; `row_factors[:, None] * matrix`
```

**When to use it.** Use it for per-column or per-row scaling when shapes and meanings are explicit.

**When to use another approach.** Avoid accidental alignment; print both operand shapes before trusting the result.

In [ ]:
# Demonstration — Broadcasting
topic_raw = np.array([[10.0, 20.0], [30.0, 40.0], [50.0, 60.0]])
topic_column_scale = np.array([0.1, 2.0])
topic_scaled = topic_raw * topic_column_scale
print("shapes:", topic_raw.shape, topic_column_scale.shape, topic_scaled.shape)
print(topic_scaled, sep=chr(10))

**Expected output pattern and interpretation.** The two factors apply to their corresponding columns across all three rows, and the output retains shape `(3, 2)`.

**Science-communication statement.** Explain which physical channel each factor belongs to; matching shapes alone do not guarantee correct calibration.

### 6.2 Individual topic — Vectorization

**What it is and how it works.** Vectorization expresses elementwise or array-level computation without an explicit Python loop, delegating work to optimized NumPy operations.

**Core syntax**

```python
`result = values * scale + offset`
```

**When to use it.** Use it for uniform numerical operations over compatible arrays.

**When to use another approach.** Avoid compressing multi-step validation or stateful logic into unreadable vector expressions.

In [ ]:
# Demonstration — Vectorization
topic_values = np.array([1.0, 2.0, 3.0])
topic_vectorized = topic_values ** 2 + 1
topic_loop = np.array([value ** 2 + 1 for value in topic_values])
print("vectorized:", topic_vectorized)
print("same as loop:", np.array_equal(topic_vectorized, topic_loop))

**Expected output pattern and interpretation.** Both approaches return `[2, 5, 10]`; the equality check verifies the teaching example's equivalence.

**Science-communication statement.** Describe the operation and array population, not merely that vectorization is faster; performance depends on data size and context.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** Broadcasting is not magic duplication; it is a documented alignment rule.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Apply one calibration offset per sensor column to a reading matrix through broadcasting, then verify the vectorized result against an explicit loop.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Broadcasting and vectorized computation
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 7. Case study: power and quality calculations

Suppose the first two columns of an array contain voltage and current. Power is computed elementwise, and a quality rule marks readings outside plausible ranges.


In [ ]:
electrical = np.array(
    [
        [12.1, 1.20],
        [11.9, 1.35],
        [12.3, 1.10],
        [18.0, 9.50],
        [12.0, 1.28],
    ]
)

voltage_v = electrical[:, 0]
current_a = electrical[:, 1]
power_w = voltage_v * current_a
valid = (voltage_v >= 10) & (voltage_v <= 15) & (current_a >= 0) & (current_a <= 5)

report = {
    "power_w": np.round(power_w, 2).tolist(),
    "valid_rows": int(valid.sum()),
    "rejected_rows": int((~valid).sum()),
    "mean_valid_power_w": round(float(power_w[valid].mean()), 2),
}
report


### Discussion clinic — vectorization still requires validation and denominators

The case study computes every row's power at once, but speed is not the only goal. The voltage and current masks state admissible evidence, `valid.sum()` counts retained rows, and the complement counts rejections. The mean uses only retained power values. Report the denominator explicitly: 'mean of four valid rows' is materially different from 'mean of five received rows.' An empty valid selection must be handled before calling `.mean()`.

**Interpretation standard.** A strong NumPy report includes shapes, units, validation rules, retained and rejected counts, and a numerical range check. Vectorization reduces loop syntax but does not remove those responsibilities.

### Science communication lens

- **Audience:** A reader who needs the numerical result and enough array structure to reproduce it.
- **Lead with the meaning:** A strong NumPy report includes shapes, units, validation rules, retained and rejected counts, and a numerical range check.
- **Show the evidence:** Report shape, dtype, axis meaning, units, valid and rejected counts, and a plausible numerical range—not only the printed array.
- **State the boundary:** Vectorized computation can be fast and correct relative to the code while still using the wrong axis, unit, calibration, or validation rule.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Calculate apparent power from paired voltage and current arrays, reject physically implausible or missing inputs, and report both accepted powers and rejection reasons.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Case study: power and quality calculations
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## Common mistakes

- Assuming `*` means matrix multiplication; for arrays it is elementwise. Use `@` for matrix products.
- Combining masks with `and` instead of `&`.
- Ignoring shape after slicing or reshaping.
- Accidentally modifying an original array through a view.
- Allowing integer dtype to truncate a calculation that should be floating point.


In [ ]:
# Guided check: normalize each column to a 0-1 range.
mins = raw.min(axis=0)
spans = raw.max(axis=0) - mins
normalized = (raw - mins) / spans

assert normalized.shape == raw.shape
assert np.allclose(normalized.min(axis=0), 0)
assert np.allclose(normalized.max(axis=0), 1)
np.round(normalized, 3)


## Independent challenge

Create a `(12, 4)` array representing twelve observations from four sensors. Apply a different scale and offset to each sensor, flag implausible values with a Boolean mask, calculate per-sensor summaries, and explain which axis each aggregation uses.

**Submission expectation:** include readable code, meaningful variable names, a short interpretation, and evidence that the notebook was restarted and run from top to bottom.


## Key takeaways

            - Array shape and dtype are part of the meaning of numerical data.
- Indexing, masks, and reshaping should be followed by explicit checks.
- Broadcasting expresses repeated arithmetic compactly when shapes are compatible.
- Vectorization improves clarity when the operation truly applies elementwise.

            ## Exit ticket

            1. What is the difference between a NumPy view and copy?
2. Why must each comparison be parenthesized when combining masks?
3. What real-world meaning would you assign to the two axes of a `(24, 3)` array?


## References and further reading

 - NumPy Developers. NumPy User Guide: Array creation, indexing, broadcasting, and statistics.
- VanderPlas, J. (2022). Python Data Science Handbook (2nd ed.).
- CPE15 syllabus, Week 2 course learning plan.

<details>
   <summary><strong>Instructor facilitation note</strong></summary>

            Ask students to predict outputs before execution, compare at least two valid approaches, and explain results in plain language. During live coding, deliberately trigger one common error and model a calm debugging process.
 </details>


In [1]:
import numpy as np

def print_separator():
    print("\n- - - - - - - - - - - - - -\n")

raw_readings = np.array([
    [512, 208, 715, 1005],
    [518, 203, 728, 1012],
    [505, 210, 705,  998],
    [522, 199, 733, 1020],
    [509, 206, 718, 1003],
    [515, 990, 722, 1008],
    [520, 201, 710,  995],
    [507, 207, 726, 1015],
    [513, 204, 719, 1006],
    [519, 202, 731, 1018],
    [504, 209, 708,  999],
    [516, 205, 720,   15],
], dtype=float)

readings_scale_factor = np.array([0.0125, 0.005, 0.0075, 0.08])
readings_offset = np.array([0.02, 0.1, -0.1, 0.2])

calibrated_readings = (raw_readings * readings_scale_factor) + readings_offset

print_separator();
print("Calibrated readings: \n")
print(calibrated_readings)
print_separator();

lower_bounds = np.array([5.0, 1.0, 5.0, 70.0])
upper_bounds = np.array([7.0, 4.0, 6.0, 90.0])

valid_reading_mask = (calibrated_readings >= lower_bounds) & (calibrated_readings <= upper_bounds)

print_separator()
print("Valid Readings")
print(valid_reading_mask)
print_separator()

valid_rows = valid_reading_mask.sum(axis=0)
rejected_rows = (~valid_reading_mask).sum(axis=0)
mean_readings = calibrated_readings.mean(axis=0)
std_readings = calibrated_readings.std(axis=0)

report = {
    "Valid readings" : valid_rows.tolist(),
    "Rejected readings" : rejected_rows.tolist(),
    "Statistics": {
        "Mean" : mean_readings,
        "Standard Deviation": std_readings
    }
}

actual_total_readings = calibrated_readings.sum(axis=0)
rejected_total_readings = np.where(~valid_reading_mask, calibrated_readings, 0).sum(axis=0)
accepted_total_readings = np.where(valid_reading_mask, calibrated_readings, 0).sum(axis=0)

collected_total_readings = accepted_total_readings + rejected_total_readings

assert np.array_equal(actual_total_readings, collected_total_readings)

print_separator()
report


- - - - - - - - - - - - - -

Calibrated readings: 

[[ 6.42    1.14    5.2625 80.6   ]
 [ 6.495   1.115   5.36   81.16  ]
 [ 6.3325  1.15    5.1875 80.04  ]
 [ 6.545   1.095   5.3975 81.8   ]
 [ 6.3825  1.13    5.285  80.44  ]
 [ 6.4575  5.05    5.315  80.84  ]
 [ 6.52    1.105   5.225  79.8   ]
 [ 6.3575  1.135   5.345  81.4   ]
 [ 6.4325  1.12    5.2925 80.68  ]
 [ 6.5075  1.11    5.3825 81.64  ]
 [ 6.32    1.145   5.21   80.12  ]
 [ 6.47    1.125   5.3     1.4   ]]

- - - - - - - - - - - - - -


- - - - - - - - - - - - - -

Valid Readings
[[ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True False  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True False]]

- - - - - - - - - - - - - -


- - - - - - - - - - - - - -



{'Valid readings': [12, 11, 12, 11],
 'Rejected readings': [0, 1, 0, 1],
 'Statistics': {'Mean': array([ 6.43666667,  1.45166667,  5.296875  , 74.16      ]),
  'Standard Deviation': array([ 0.0722289 ,  1.0850544 ,  0.06462331, 21.9463467 ])}}